In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.utils.class_weight import compute_class_weight

import joblib
import os

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/processed/banking_fraud_featured.csv")

print(df.shape)

df.head()

(590540, 478)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,IsYahoo,IsOutlook,RoundedAmount,DecimalAmount,AboveGlobalMean,CardProductFreq,AddrProductFreq,EmailDeviceFreq,NormalizedAmount,NormalizedDistance
0,2987000.0,0,86400.0,68.5,W,13926.0,361.0,150.0,discover,142.0,...,0,0,0,1,0,31,20224,295954,-0.278167,-0.138579
1,2987001.0,0,86401.0,29.0,W,2755.0,404.0,150.0,mastercard,102.0,...,0,0,0,0,0,619,37520,295954,-0.443327,-0.183967
2,2987002.0,0,86469.0,59.0,W,4663.0,490.0,150.0,visa,166.0,...,0,1,0,0,0,1098,18411,4078,-0.317889,0.967245
3,2987003.0,0,86499.0,50.0,W,18132.0,567.0,150.0,mastercard,117.0,...,1,0,0,0,0,3904,8383,95778,-0.355520,-0.183967
4,2987004.0,0,86506.0,50.0,H,4497.0,514.0,150.0,mastercard,102.0,...,0,0,0,0,0,4,418,26857,-0.355520,-0.183967


In [3]:
X = df.drop(columns=["isFraud"])

y = df["isFraud"]

print(X.shape)

print(y.shape)

(590540, 477)
(590540,)


In [4]:
print(y.value_counts())

print()

print(y.value_counts(normalize=True)*100)

isFraud
0    569877
1     20663
Name: count, dtype: int64

isFraud
0    96.500999
1     3.499001
Name: proportion, dtype: float64


In [5]:
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical:",len(categorical_columns))

print("Numerical:",len(numerical_columns))

Categorical: 29
Numerical: 448


In [6]:
label_encoders = {}

for col in categorical_columns:

    encoder = LabelEncoder()

    X[col] = encoder.fit_transform(X[col].astype(str))

    label_encoders[col] = encoder

In [7]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print(X_train.shape)

print(X_test.shape)

(472432, 477)
(118108, 477)


In [8]:
classes = np.unique(y_train)

weights = compute_class_weight(

    class_weight="balanced",

    classes=classes,

    y=y_train

)

class_weights = {

    classes[i]:weights[i]

    for i in range(len(classes))

}

print(class_weights)

{np.int64(0): np.float64(0.5181288961224123), np.int64(1): np.float64(14.290139140955837)}


In [10]:
models = {

    "Logistic Regression": LogisticRegression(

        max_iter=1000,

        class_weight=class_weights,

        random_state=42

    ),

    "Decision Tree": DecisionTreeClassifier(

        class_weight=class_weights,

        random_state=42

    ),

    "Random Forest": RandomForestClassifier(

        n_estimators=300,

        class_weight=class_weights,

        random_state=42,

        n_jobs=-1

    ),

    "XGBoost": XGBClassifier(

        random_state=42,

        eval_metric="logloss",

        n_estimators=300,

        learning_rate=0.05,

        max_depth=8,

        subsample=0.8,

        colsample_bytree=0.8

    ),

    "LightGBM": LGBMClassifier(

        random_state=42,

        n_estimators=300,

        learning_rate=0.05

    )

}

In [11]:
trained_models = {}

for name, model in models.items():

    print("="*60)

    print(f"Training {name}")

    model.fit(

        X_train,

        y_train

    )

    trained_models[name] = model

    print(f"{name} Trained Successfully")

Training Logistic Regression
Logistic Regression Trained Successfully
Training Decision Tree
Decision Tree Trained Successfully
Training Random Forest
Random Forest Trained Successfully
Training XGBoost
XGBoost Trained Successfully
Training LightGBM
[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.316341 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 44325
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 473
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101
LightGBM Trained Successfully


In [12]:
os.makedirs("../models",exist_ok=True)

In [13]:
for name,model in trained_models.items():

    filename = name.lower().replace(" ","_") + ".pkl"

    joblib.dump(

        model,

        "../models/"+filename

    )

print("Models Saved Successfully")

Models Saved Successfully


In [14]:
joblib.dump(

    label_encoders,

    "../models/label_encoders.pkl"

)

print("Encoders Saved Successfully")

Encoders Saved Successfully


In [15]:
joblib.dump(

    X_train,

    "../models/X_train.pkl"

)

joblib.dump(

    X_test,

    "../models/X_test.pkl"

)

joblib.dump(

    y_train,

    "../models/y_train.pkl"

)

joblib.dump(

    y_test,

    "../models/y_test.pkl"

)

['../models/y_test.pkl']

In [16]:
os.listdir("../models")

['decision_tree.pkl',
 'label_encoders.pkl',
 'lightgbm.pkl',
 'logistic_regression.pkl',
 'random_forest.pkl',
 'xgboost.pkl',
 'X_test.pkl',
 'X_train.pkl',
 'y_test.pkl',
 'y_train.pkl']